# Threshold & Window Size Optimization
Grid search over K_VISUAL, K_MV, K_AUDIO_RMS, K_FLUX thresholds + MERGE_WINDOW & SELECTION_WINDOW sizes.
Plots Hit Rate vs FP for each parameter sweep.

In [1]:
import json, os, time, itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque
from tqdm.notebook import tqdm

os.makedirs('plots', exist_ok=True)
SCRIPT_DIR = os.getcwd()
WORKER_DATA_DIR = 'worker_data'
GROUND_TRUTH_JSON = 'attack.json'
VIDEOS_TO_MONITOR = [f'attacked_{i}' for i in range(1, 42)]

In [2]:
# ── Defaults (untouched unless overridden) ──────────────────────────────────
DEF = dict(
    BOUNDARY_TOLERANCE=2.5, TOLERANCE_SEC=2.5, LOCKOUT_PERIOD=5.0,
    MERGE_WINDOW=10.0, SELECTION_WINDOW_SEC=30.0, SELECTION_OVERLAP=0.1, SELECTION_MIN_GAP=0.5,
    VISUAL_MAX_FRAMES=30, VISUAL_MIN_WARMUP=10, K_VISUAL_THRESHOLD=2.5,
    VISUAL_COOLDOWN=3.0, VISUAL_PERSIST_NEEDED=2, VISUAL_PERSIST_WINDOW=4,
    SCENE_CHANGE_RATIO=8.0, SCENE_CHANGE_TOLERANCE=0.5,
    IFRAME_SANDWICH_ENABLED=True, IFRAME_CONTEXT_WINDOW=5,
    IFRAME_STABILITY_RATIO=3.0, IFRAME_MIN_CONTEXT=3,
    MV_WINDOW=60, MV_MIN_WARMUP=30, K_MV_THRESHOLD=4.5,
    MV_COOLDOWN=5.0, MV_PERSIST_NEEDED=3, MV_PERSIST_WINDOW=5,
    GOP_SHORT_RATIO=0.25, MOTION_JERK_ENABLED=True,
    JERK_WINDOW=5, JERK_ACCEL_THRESHOLD=2.5,
    AUDIO_SAMPLE_RATE=44100, AUDIO_BUFFER_SEC=15.0,
    AUDIO_MICRO_WINDOW_SEC=0.5, K_AUDIO_RMS_THRESHOLD=3.0,
    AUDIO_NOISE_FLOOR=800.0, AUDIO_COOLDOWN=3.0, AUDIO_PERSIST_NEEDED=2,
    SILENCE_THRESHOLD_RATIO=0.05, SILENCE_MIN_DURATION=2.0, SILENCE_COOLDOWN=30.0,
    FLUX_SAMPLE_RATE=22050, FLUX_WINDOW_SEC=0.05, FLUX_BUFFER_SEC=10.0,
    K_FLUX_THRESHOLD=3.0, FLUX_COOLDOWN=3.0, FLUX_PERSIST_NEEDED=3,
    MIN_SIGNALS_TO_CONFIRM=1, REQUIRE_MULTIMODAL=False,
    AUDIO_UPGRADE_WINDOW=2.5, MIN_EVENT_GAP=0.5,
    W_CONFIDENCE=0.50, W_MAGNITUDE=0.05, W_ACCELERATION=0.15,
    W_EDGE_CHANGE=0.10, W_UNIFORMITY=0.05, W_PERSISTENCE=0.15,
)
VISUAL_MOTION_SIGNALS = {'Visual', 'Motion'}
AUDIO_SIGNALS = {'AudioRMS', 'Silence', 'SpectralFlux'}
ALL_SIGNALS = ['Visual', 'Motion', 'AudioRMS', 'Silence', 'SpectralFlux']
print('Defaults loaded.')

Defaults loaded.


In [3]:
# ── Core detection logic (parameterised) ────────────────────────────────────
def adaptive_zscore(value, buffer, k):
    arr = np.array(buffer, dtype=np.float64)
    if len(arr) < 2: return 0.0, False
    mu, sigma = np.mean(arr), np.std(arr)
    if sigma < 1e-9: return 0.0, False
    z = (value - mu) / sigma
    return z, abs(z) > k

def confidence_score(w):
    return 1.0 if w >= 3 else (0.65 if w == 2 else 0.30)

def compute_trigger_score(weight, feats, p):
    return float(np.clip(
        p['W_CONFIDENCE']*confidence_score(weight)
        + p['W_MAGNITUDE']*feats.get('magnitude',0.5)
        + p['W_ACCELERATION']*feats.get('acceleration',0.5)
        + p['W_EDGE_CHANGE']*feats.get('edge_change',0.5)
        + p['W_UNIFORMITY']*feats.get('uniformity',0.5)
        + p['W_PERSISTENCE']*feats.get('persistence',0.5), 0.0, 1.0))

def run_detection(packets, p):
    frames, audio_samples, flux_samples = [], [], []
    for pkt in packets:
        st, fps = pkt['chunk_start_time'], pkt['fps']
        for idx, frm in enumerate(pkt['frames']):
            frames.append((st + idx/fps, frm['type'], int(frm['size'])))
        for r in pkt['audio_rms']:
            if r['time'] >= st - 0.001: audio_samples.append((r['time'], r['rms']))
        for f in pkt['spectral_flux']:
            if f['time'] >= st - 0.001: flux_samples.append((f['time'], f['flux']))
    frames.sort(key=lambda x: x[0])
    audio_samples.sort(key=lambda x: x[0])
    flux_samples.sort(key=lambda x: x[0])

    # Visual + Motion
    v_size_buf = deque(maxlen=p['VISUAL_MAX_FRAMES'])
    v_spike_flags = deque(maxlen=p['VISUAL_PERSIST_WINDOW'])
    v_last_trig = -999.0; v_triggers = []; v_sizes_sandwich = []
    scene_trans_end = 0.0
    mv_pb_buf = deque(maxlen=p['MV_WINDOW'])
    mv_spike_flags = deque(maxlen=p['MV_PERSIST_WINDOW'])
    mv_last_trig = -999.0; mv_last_i_t = -999.0
    mv_gop = deque(maxlen=30); mv_triggers = []
    jerk_history = deque(maxlen=p['JERK_WINDOW'])

    for (t, ptype, size) in frames:
        is_i = (ptype == 'I')
        if is_i:
            v_sizes_sandwich.append((t, size))
            cur_idx = len(v_sizes_sandwich)-1
            if len(v_size_buf) >= 2:
                prev = list(v_size_buf)[-1]
                if prev > 0 and size/prev > p['SCENE_CHANGE_RATIO']:
                    scene_trans_end = t + p['SCENE_CHANGE_TOLERANCE']
            spike = False
            if t >= scene_trans_end and t > p['LOCKOUT_PERIOD'] and len(v_size_buf) >= p['VISUAL_MIN_WARMUP']:
                _, spike = adaptive_zscore(size, v_size_buf, p['K_VISUAL_THRESHOLD'])
            v_spike_flags.append(1 if spike else 0)
            if (t > p['LOCKOUT_PERIOD'] and len(v_spike_flags) == p['VISUAL_PERSIST_WINDOW']
                    and sum(v_spike_flags) >= p['VISUAL_PERSIST_NEEDED']
                    and (t - v_last_trig) > p['VISUAL_COOLDOWN']
                    and (t - v_last_trig) > p['MIN_EVENT_GAP']):
                v_triggers.append(t); v_last_trig = t; v_spike_flags.clear()
            v_size_buf.append(size)
            if mv_last_i_t > 0:
                interval = t - mv_last_i_t
                if len(mv_gop) >= 10:
                    mean_gop = np.mean(mv_gop)
                    if (t > p['LOCKOUT_PERIOD'] and mean_gop > 0
                            and interval < mean_gop*p['GOP_SHORT_RATIO']
                            and (t - mv_last_trig) > p['MV_COOLDOWN']):
                        mv_triggers.append(t); mv_last_trig = t
                mv_gop.append(interval)
            mv_last_i_t = t
        if ptype in ('P','B'):
            jerk_history.append(size); mv_pb_buf.append(size)
            spike = False
            if t > p['LOCKOUT_PERIOD'] and len(mv_pb_buf) >= p['MV_MIN_WARMUP']:
                _, spike = adaptive_zscore(size, mv_pb_buf, p['K_MV_THRESHOLD'])
            mv_spike_flags.append(1 if spike else 0)
            if (t > p['LOCKOUT_PERIOD'] and len(mv_spike_flags) == p['VISUAL_PERSIST_WINDOW']
                    and sum(mv_spike_flags) >= p['VISUAL_PERSIST_NEEDED']
                    and (t - v_last_trig) > p['VISUAL_COOLDOWN']
                    and (t - v_last_trig) > p['MIN_EVENT_GAP']):
                mv_triggers.append(t); mv_last_trig = t; mv_spike_flags.clear()

    mv_triggers.sort()
    deduped_mv, last_t = [], -999.0
    for t in mv_triggers:
        if t - last_t > p['MV_COOLDOWN']: deduped_mv.append(t); last_t = t

    # AudioRMS + Silence
    rms_buf = deque(maxlen=max(1, int(p['AUDIO_BUFFER_SEC']/p['AUDIO_MICRO_WINDOW_SEC'])))
    rms_consec = 0; rms_last_trig = -999.0; rms_triggers = []
    sil_in = False; sil_start = -999.0; sil_last_trig = -999.0; sil_triggers = []
    for (micro_t, micro_rms) in audio_samples:
        if micro_t > p['LOCKOUT_PERIOD'] and len(rms_buf) == rms_buf.maxlen:
            mu = float(np.mean(rms_buf)); sigma = float(np.std(rms_buf))
            is_spike = (micro_rms > p['AUDIO_NOISE_FLOOR'] and sigma > 0
                        and micro_rms > mu + p['K_AUDIO_RMS_THRESHOLD']*sigma)
            rms_consec = rms_consec+1 if is_spike else 0
            if (rms_consec >= p['AUDIO_PERSIST_NEEDED']
                    and (micro_t - rms_last_trig) > p['AUDIO_COOLDOWN']
                    and (micro_t - rms_last_trig) > p['MIN_EVENT_GAP']):
                rms_triggers.append(micro_t); rms_last_trig = micro_t; rms_consec = 0
            if mu > p['AUDIO_NOISE_FLOOR']:
                is_silent = micro_rms < mu*p['SILENCE_THRESHOLD_RATIO']
                if is_silent and not sil_in: sil_in, sil_start = True, micro_t
                elif not is_silent and sil_in:
                    sil_in = False
                    dur = micro_t - sil_start
                    if dur >= p['SILENCE_MIN_DURATION'] and (sil_start - sil_last_trig) > p['SILENCE_COOLDOWN']:
                        sil_triggers.append(sil_start); sil_last_trig = sil_start
        rms_buf.append(micro_rms)

    # SpectralFlux
    flux_buf = deque(maxlen=max(10, int(p['FLUX_BUFFER_SEC']/p['FLUX_WINDOW_SEC'])))
    flux_consec = 0; flux_last_trig = -999.0; flux_triggers = []
    for (flux_t, fv) in flux_samples:
        if flux_t > p['LOCKOUT_PERIOD'] and len(flux_buf) >= int(flux_buf.maxlen*0.3):
            _, spike = adaptive_zscore(fv, flux_buf, p['K_FLUX_THRESHOLD'])
            flux_consec = flux_consec+1 if spike else 0
            if (flux_consec >= p['FLUX_PERSIST_NEEDED']
                    and (flux_t - flux_last_trig) > p['FLUX_COOLDOWN']
                    and (flux_t - flux_last_trig) > p['MIN_EVENT_GAP']):
                flux_triggers.append(flux_t); flux_last_trig = flux_t; flux_consec = 0
        flux_buf.append(fv)

    return {'Visual': v_triggers, 'Motion': deduped_mv,
            'AudioRMS': rms_triggers, 'Silence': sil_triggers, 'SpectralFlux': flux_triggers}

print('Detection function defined.')

Detection function defined.


In [4]:
# ── Fusion + selection (parameterised) ──────────────────────────────────────
def fuse_all_signals(signal_dict, p):
    all_events = sorted([{'time':t,'signal':lbl} for lbl,ts in signal_dict.items() for t in ts], key=lambda x: x['time'])
    if not all_events: return []
    used = [False]*len(all_events); clusters = []
    for i, ev in enumerate(all_events):
        if used[i]: continue
        cluster = [ev]; used[i] = True
        for j in range(i+1, len(all_events)):
            if used[j]: continue
            if all_events[j]['time'] - cluster[0]['time'] > p['TOLERANCE_SEC']: break
            cluster.append(all_events[j]); used[j] = True
        clusters.append(cluster)
    merged = []
    for cluster in clusters:
        sigs = list({e['signal'] for e in cluster})
        vm_sigs = [s for s in sigs if s in VISUAL_MOTION_SIGNALS]
        if not vm_sigs: continue
        avg_t = round(float(np.mean([e['time'] for e in cluster])), 2)
        base_weight = len(set(vm_sigs))
        audio_present = {e['signal'] for e in cluster if e['signal'] in AUDIO_SIGNALS and abs(e['time']-avg_t) <= p['AUDIO_UPGRADE_WINDOW']}
        weight = base_weight + len(audio_present)
        all_sigs = sorted(set(vm_sigs)|audio_present)
        feats = {k:0.5 for k in ['magnitude','acceleration','edge_change','uniformity','persistence']}
        score = compute_trigger_score(weight, feats, p)
        merged.append((avg_t, '+'.join(all_sigs), len(all_sigs), min(weight,5), feats, score))
    return sorted(merged, key=lambda x: x[0])

def apply_window_merge(fused_events, p):
    if not fused_events: return []
    result, i = [], 0
    while i < len(fused_events):
        group, j = [], i
        while j < len(fused_events) and fused_events[j][0] <= fused_events[i][0]+p['MERGE_WINDOW']:
            group.append(fused_events[j]); j += 1
        if len(group) >= 2:
            all_lbl = set(s for e in group for s in e[1].split('+'))
            feats = {k:0.5 for k in ['magnitude','acceleration','edge_change','uniformity','persistence']}
            score = compute_trigger_score(3, feats, p)
            result.append((round(float(np.mean([e[0] for e in group])),2), '+'.join(sorted(all_lbl)), len(all_lbl), 3, feats, score))
        else: result.append(group[0])
        i = j
    return result

def select_strongest_per_window(event_dicts, p):
    if not event_dicts: return [], []
    window_sec = p['SELECTION_WINDOW_SEC']
    overlap    = p['SELECTION_OVERLAP']
    min_gap    = p['SELECTION_MIN_GAP']
    hop_sec    = window_sec*(1.0-overlap)
    total_time = max(e['time'] for e in event_dicts)+1.0
    n_windows  = max(1, int(np.ceil((total_time-window_sec)/hop_sec))+1)
    kept_set   = set()
    for w in range(n_windows):
        ws, we = w*hop_sec, w*hop_sec+window_sec
        in_win = [i for i,e in enumerate(event_dicts) if ws <= e['time'] < we]
        if in_win: kept_set.add(max(in_win, key=lambda i: event_dicts[i]['score']))
    sorted_kept = sorted(kept_set, key=lambda i: event_dicts[i]['time'])
    final_kept, last_t = [], -999.0
    for i in sorted_kept:
        t = event_dicts[i]['time']
        if t-last_t >= min_gap: final_kept.append(i); last_t = t
    return [event_dicts[i] for i in range(len(event_dicts)) if i in kept_set], \
           [event_dicts[i] for i in range(len(event_dicts)) if i not in kept_set]

def validate_video(segments, final_events, p):
    segs = [dict(s, hit=None) for s in segments]
    fp_events = []
    for ev in final_events:
        ts, label, n_sig, weight, feats, score = ev
        claimed = False
        for seg in segs:
            ns = abs(ts-seg['start']) <= p['BOUNDARY_TOLERANCE']
            ne = abs(ts-seg['end'])   <= p['BOUNDARY_TOLERANCE']
            inside = seg['start'] < ts < seg['end']
            if (ns or ne or inside) and seg['hit'] is None:
                seg['hit'] = {'ts':ts,'label':label}; claimed = True; break
        if not claimed: fp_events.append({'ts':ts,'label':label})
    return segs, fp_events

def evaluate_one_video(video_id, gt_entry, p):
    json_path = os.path.join(SCRIPT_DIR, WORKER_DATA_DIR, f'{video_id}_worker_data.json')
    if not os.path.exists(json_path): return None
    with open(json_path) as f: data = json.load(f)
    packets = data['chunks']
    signal_dict  = run_detection(packets, p)
    fused        = fuse_all_signals(signal_dict, p)
    merged       = apply_window_merge(fused, p)
    event_dicts  = [{'time':e[0],'label':e[1],'n_sig':e[2],'weight':e[3],'features':e[4],'score':e[5]} for e in merged]
    selected, _  = select_strongest_per_window(event_dicts, p)
    final_events = [(e['time'],e['label'],e['n_sig'],e['weight'],e['features'],e['score']) for e in selected]
    segs, fp_events = validate_video(gt_entry['segments'], final_events, p)
    hits = sum(1 for s in segs if s['hit'])
    return {'total': len(gt_entry['segments']), 'hits': hits, 'fp': len(fp_events)}

def evaluate_all(override: dict) -> dict:
    p = {**DEF, **override}
    total_segs = hits = fp = 0
    for vid in VIDEOS_TO_MONITOR:
        vf = f'{vid}.mp4'
        if vf not in gt_map: continue
        r = evaluate_one_video(vid, gt_map[vf], p)
        if r is None: continue
        total_segs += r['total']; hits += r['hits']; fp += r['fp']
    hr = hits/total_segs*100 if total_segs else 0
    return {'hit_rate': hr, 'hits': hits, 'fp': fp, 'total': total_segs}

print('Evaluation functions defined.')

Evaluation functions defined.


In [5]:
# ── Load ground truth ────────────────────────────────────────────────────────
gt_path = os.path.join(SCRIPT_DIR, GROUND_TRUTH_JSON)
with open(gt_path) as f: ground_truth = json.load(f)
gt_map = {e['target_video']: e for e in ground_truth}
print(f'Loaded {len(gt_map)} ground-truth entries.')

Loaded 40 ground-truth entries.


## 1 – K-Threshold Sweeps (Visual · Motion · AudioRMS · SpectralFlux)

In [ ]:
K_RANGE = np.round(np.arange(1.0, 7.5, 0.5), 1).tolist()

THRESHOLD_PARAMS = {
    'K_VISUAL_THRESHOLD':    'K_VISUAL_THRESHOLD',
    'K_MV_THRESHOLD':        'K_MV_THRESHOLD',
    'K_AUDIO_RMS_THRESHOLD': 'K_AUDIO_RMS_THRESHOLD',
    'K_FLUX_THRESHOLD':      'K_FLUX_THRESHOLD',
}

thresh_results = {}
total_runs = len(THRESHOLD_PARAMS) * len(K_RANGE)

with tqdm(total=total_runs, desc='Threshold sweep') as pbar:
    for param_name in THRESHOLD_PARAMS:
        rows = []
        for k in K_RANGE:
            r = evaluate_all({param_name: k})
            rows.append({'k': k, 'hit_rate': r['hit_rate'], 'fp': r['fp']})
            pbar.update(1)
        thresh_results[param_name] = rows

print('Threshold sweep complete.')

Threshold sweep:   0%|          | 0/52 [00:00<?, ?it/s]

In [ ]:
# ── Plot: Hit Rate & FP vs K (one subplot per threshold) ────────────────────
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
fig.suptitle('K-Threshold Sweep: Hit Rate & False Positives', fontsize=15, fontweight='bold', y=1.01)

COLORS = {'hit_rate': '#2196F3', 'fp': '#F44336'}
LABELS = {
    'K_VISUAL_THRESHOLD':    'K_VISUAL',
    'K_MV_THRESHOLD':        'K_MV',
    'K_AUDIO_RMS_THRESHOLD': 'K_AUDIO_RMS',
    'K_FLUX_THRESHOLD':      'K_FLUX',
}

for row, param_name in enumerate(THRESHOLD_PARAMS):
    rows = thresh_results[param_name]
    ks   = [r['k']        for r in rows]
    hrs  = [r['hit_rate'] for r in rows]
    fps  = [r['fp']       for r in rows]

    # Hit Rate
    ax = axes[row][0]
    ax.plot(ks, hrs, 'o-', color=COLORS['hit_rate'], lw=2, ms=6)
    ax.fill_between(ks, hrs, alpha=0.12, color=COLORS['hit_rate'])
    best_k = ks[np.argmax(hrs)]
    ax.axvline(best_k, ls='--', color='grey', lw=1, label=f'Best k={best_k}')
    ax.set_title(f'{LABELS[param_name]} – Hit Rate %', fontsize=11)
    ax.set_xlabel('K value'); ax.set_ylabel('Hit Rate (%)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 105)

    # FP
    ax = axes[row][1]
    ax.plot(ks, fps, 's-', color=COLORS['fp'], lw=2, ms=6)
    ax.fill_between(ks, fps, alpha=0.12, color=COLORS['fp'])
    best_k_fp = ks[np.argmin(fps)]
    ax.axvline(best_k_fp, ls='--', color='grey', lw=1, label=f'Min FP k={best_k_fp}')
    ax.set_title(f'{LABELS[param_name]} – False Positives', fontsize=11)
    ax.set_xlabel('K value'); ax.set_ylabel('Total FP')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/threshold_sweep.png')

In [ ]:
# ── Scatter: Hit Rate vs FP per threshold ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Hit Rate vs False Positives (Pareto View)', fontsize=14, fontweight='bold')

SCATTER_COLORS = ['#2196F3','#4CAF50','#FF9800','#9C27B0']

for idx, param_name in enumerate(THRESHOLD_PARAMS):
    ax   = axes[idx//2][idx%2]
    rows = thresh_results[param_name]
    ks   = [r['k']        for r in rows]
    hrs  = [r['hit_rate'] for r in rows]
    fps  = [r['fp']       for r in rows]
    sc   = ax.scatter(fps, hrs, c=ks, cmap='RdYlGn_r', s=90, zorder=3, edgecolors='k', lw=0.5)
    for k, fp, hr in zip(ks, fps, hrs):
        ax.annotate(f'k={k}', (fp, hr), textcoords='offset points', xytext=(4,4), fontsize=7)
    plt.colorbar(sc, ax=ax, label='K value')
    ax.set_title(LABELS[param_name], fontsize=11)
    ax.set_xlabel('False Positives'); ax.set_ylabel('Hit Rate (%)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/threshold_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/threshold_pareto.png')

## 2 – Window Size Sweeps (MERGE_WINDOW & SELECTION_WINDOW_SEC)

In [ ]:
MERGE_WINDOWS     = [2, 4, 6, 8, 10, 12, 15, 20, 25, 30]
SELECTION_WINDOWS = [10, 15, 20, 25, 30, 40, 50, 60, 90]

merge_rows = []
sel_rows   = []

with tqdm(total=len(MERGE_WINDOWS)+len(SELECTION_WINDOWS), desc='Window sweep') as pbar:
    for w in MERGE_WINDOWS:
        r = evaluate_all({'MERGE_WINDOW': w})
        merge_rows.append({'w': w, 'hit_rate': r['hit_rate'], 'fp': r['fp']})
        pbar.update(1)
    for w in SELECTION_WINDOWS:
        r = evaluate_all({'SELECTION_WINDOW_SEC': w})
        sel_rows.append({'w': w, 'hit_rate': r['hit_rate'], 'fp': r['fp']})
        pbar.update(1)

print('Window sweep complete.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Window Size Sweep: Hit Rate & False Positives', fontsize=14, fontweight='bold')

for col, (rows, title) in enumerate([
        (merge_rows, 'MERGE_WINDOW'), (sel_rows, 'SELECTION_WINDOW_SEC')]):
    ws  = [r['w']        for r in rows]
    hrs = [r['hit_rate'] for r in rows]
    fps = [r['fp']       for r in rows]

    ax = axes[0][col]
    ax.plot(ws, hrs, 'o-', color='#2196F3', lw=2, ms=7)
    ax.fill_between(ws, hrs, alpha=0.12, color='#2196F3')
    ax.axvline(ws[np.argmax(hrs)], ls='--', color='grey', lw=1, label=f'Best={ws[np.argmax(hrs)]}s')
    ax.set_title(f'{title} – Hit Rate %'); ax.set_xlabel('Window (s)'); ax.set_ylabel('Hit Rate (%)')
    ax.set_ylim(0,105); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[1][col]
    ax.plot(ws, fps, 's-', color='#F44336', lw=2, ms=7)
    ax.fill_between(ws, fps, alpha=0.12, color='#F44336')
    ax.axvline(ws[np.argmin(fps)], ls='--', color='grey', lw=1, label=f'Min FP={ws[np.argmin(fps)]}s')
    ax.set_title(f'{title} – False Positives'); ax.set_xlabel('Window (s)'); ax.set_ylabel('Total FP')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/window_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/window_sweep.png')

In [ ]:
# Window Pareto scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Window Size – Hit Rate vs FP (Pareto View)', fontsize=13, fontweight='bold')

for ax, rows, title in zip(axes,
        [merge_rows, sel_rows],
        ['MERGE_WINDOW', 'SELECTION_WINDOW_SEC']):
    ws  = [r['w']        for r in rows]
    hrs = [r['hit_rate'] for r in rows]
    fps = [r['fp']       for r in rows]
    sc  = ax.scatter(fps, hrs, c=ws, cmap='viridis', s=100, zorder=3, edgecolors='k', lw=0.5)
    for w, fp, hr in zip(ws, fps, hrs):
        ax.annotate(f'{w}s', (fp, hr), textcoords='offset points', xytext=(4,4), fontsize=8)
    plt.colorbar(sc, ax=ax, label='Window (s)')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('False Positives'); ax.set_ylabel('Hit Rate (%)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/window_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/window_pareto.png')

## 3 – Grid Search: Best K Combination (Visual × Motion × Audio × Flux)

In [ ]:
# Coarse grid – refine ranges after inspecting individual sweeps above
GRID_K_VIS   = [1.5, 2.0, 2.5, 3.0, 3.5]
GRID_K_MV    = [3.0, 3.5, 4.0, 4.5, 5.0]
GRID_K_AUDIO = [2.0, 2.5, 3.0, 3.5, 4.0]
GRID_K_FLUX  = [2.0, 2.5, 3.0, 3.5, 4.0]

combos = list(itertools.product(GRID_K_VIS, GRID_K_MV, GRID_K_AUDIO, GRID_K_FLUX))
print(f'Grid size: {len(combos)} combinations')

grid_rows = []
for kv, km, ka, kf in tqdm(combos, desc='Grid search'):
    r = evaluate_all({
        'K_VISUAL_THRESHOLD': kv, 'K_MV_THRESHOLD': km,
        'K_AUDIO_RMS_THRESHOLD': ka, 'K_FLUX_THRESHOLD': kf
    })
    grid_rows.append({'kv':kv,'km':km,'ka':ka,'kf':kf,
                      'hit_rate':r['hit_rate'],'fp':r['fp']})

grid_rows.sort(key=lambda x: (-x['hit_rate'], x['fp']))
print(f'\nTop 10 combinations:')
print(f"{'kv':>5} {'km':>5} {'ka':>5} {'kf':>5} {'Hit%':>7} {'FP':>5}")
for r in grid_rows[:10]:
    print(f"{r['kv']:>5} {r['km']:>5} {r['ka']:>5} {r['kf']:>5} {r['hit_rate']:>6.1f}% {r['fp']:>5}")

In [ ]:
# ── Heatmaps: fix audio+flux at best values, vary visual × motion ────────────
best = grid_rows[0]
ka_fix, kf_fix = best['ka'], best['kf']

mask = [r for r in grid_rows if r['ka']==ka_fix and r['kf']==kf_fix]

hr_mat = np.zeros((len(GRID_K_VIS), len(GRID_K_MV)))
fp_mat = np.zeros((len(GRID_K_VIS), len(GRID_K_MV)))

for r in mask:
    i = GRID_K_VIS.index(r['kv'])
    j = GRID_K_MV.index(r['km'])
    hr_mat[i,j] = r['hit_rate']
    fp_mat[i,j] = r['fp']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Grid Heatmap (K_AUDIO={ka_fix}, K_FLUX={kf_fix})', fontsize=13, fontweight='bold')

for ax, mat, title, cmap in zip(
        axes, [hr_mat, fp_mat],
        ['Hit Rate (%)', 'False Positives'],
        ['YlGn', 'YlOrRd']):
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(GRID_K_MV)));   ax.set_xticklabels(GRID_K_MV)
    ax.set_yticks(range(len(GRID_K_VIS)));  ax.set_yticklabels(GRID_K_VIS)
    ax.set_xlabel('K_MV'); ax.set_ylabel('K_VISUAL')
    ax.set_title(title)
    for i in range(len(GRID_K_VIS)):
        for j in range(len(GRID_K_MV)):
            ax.text(j, i, f'{mat[i,j]:.0f}', ha='center', va='center', fontsize=8,
                    color='black' if mat[i,j] < mat.max()*0.8 else 'white')

plt.tight_layout()
plt.savefig('plots/grid_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/grid_heatmap.png')

In [ ]:
# ── Combined summary: top-20 hit_rate vs FP scatter ─────────────────────────
top20  = grid_rows[:20]
bot20  = grid_rows[-20:]
sample = top20 + bot20

hrs_g = [r['hit_rate'] for r in sample]
fps_g = [r['fp']       for r in sample]
labels_g = [f"kv{r['kv']} km{r['km']}" for r in sample]
colors_g = ['#4CAF50']*20 + ['#F44336']*20

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(fps_g, hrs_g, c=colors_g, s=80, edgecolors='k', lw=0.5, zorder=3)
for lbl, fp, hr, col in zip(labels_g, fps_g, hrs_g, colors_g):
    ax.annotate(lbl, (fp, hr), textcoords='offset points', xytext=(4,3), fontsize=6, color='#333')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#4CAF50',label='Top 20'), Patch(color='#F44336',label='Bottom 20')])
ax.set_xlabel('False Positives'); ax.set_ylabel('Hit Rate (%)')
ax.set_title('Grid Search: Top-20 vs Bottom-20 Combinations', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/grid_top_bottom.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/grid_top_bottom.png')

## 4 – Best Configuration Summary

In [ ]:
print('='*60)
print('RECOMMENDED CONFIGURATION (highest hit rate, then min FP)')
print('='*60)
b = grid_rows[0]
print(f"  K_VISUAL_THRESHOLD    = {b['kv']}")
print(f"  K_MV_THRESHOLD        = {b['km']}")
print(f"  K_AUDIO_RMS_THRESHOLD = {b['ka']}")
print(f"  K_FLUX_THRESHOLD      = {b['kf']}")
print(f"  → Hit Rate: {b['hit_rate']:.1f}%    FP: {b['fp']}")
print()
print('Best MERGE_WINDOW    :', merge_rows[np.argmax([r['hit_rate'] for r in merge_rows])]['w'], 's')
print('Best SELECTION_WINDOW:', sel_rows  [np.argmax([r['hit_rate'] for r in sel_rows  ])]['w'], 's')
print('Plots saved in → plots/')